# Final Project Milestone 03 Walkthrough — Complex Model + Tuning + Draft Abstract

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb15_milestone_03_walkthrough_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will know how to:

1. Translate **Final Project Milestone 03**'s rubric components into a runnable Jupyter notebook.
2. Replicate your **M2 baseline** with 5- or 10-fold cross-validation and 95% confidence interval reporting.
3. Tune a **more complex model** with `GridSearchCV` / `RandomizedSearchCV` on the training fold only (test set stays locked until M4).
4. Apply the **CI-overlap rule** to decide whether the complex model has earned displacement of the M2 baseline.
5. Retrain the chosen champion on the full training fold and save **`champion_pipeline.joblib`** + **`CONFIG.json`** for M4 to load.
6. Generate every M3 §1c **required visualization** — hyperparameter-search plot, model-comparison bar chart with 95% CI error bars, feature importance, plus the regression diagnostics (predicted-vs-actual, residual, RMSE-by-quintile) OR classification diagnostics (confusion matrix, ROC, PR, optional reliability/Brier).
7. Write a polished **250-word draft abstract** using M3's six-element template.

> **No PAUSE-AND-DO exercises in this notebook.** nb15 is a demonstration walkthrough. Your M3 submission will mirror this structure on your group's own dataset; use the cells below as a template to copy, adapt, and run on your case.

---

> **📋 Submission Reminder:** Your group's M3 deliverables (`NN_complex_model.pdf` + `NN_complex_model.ipynb`) are due on Brightspace by the posted deadline. Read this notebook side-by-side with [`milestone_03_complex_model_and_abstract.md`](../_final_project/2026Summer/milestone_03_complex_model_and_abstract.md) and use the cell structure below as a template.

---

## 💼 Why This Matters

M3 is the **modeling-plus-communication payoff** of the course. By the end of M3 your group will have:

1. A **champion model** with a defensible CV 95% confidence interval.
2. A **clear comparison** to your M2 baseline showing whether the complex model has earned the right to displace the simpler one under the CI-overlap rule.
3. A polished **250-word draft abstract** that will become the lead paragraph of your M4 poster.
4. A saved **`champion_pipeline.joblib`** that M4's test-set ceremony will load EXACTLY — no silent refits between M3 and M4.

This notebook walks you through every one of those deliverables on the two demo business cases the course has used since Week 2 — **MedScreen** (classification, Wisconsin breast cancer) and **HomeValue Analytics** (regression, California Housing). Your group's M3 will follow the same workflow on your own dataset.

> **A question that often comes up here:** *"why does M4 require the joblib if I can just refit later?"* Because **refitting silently changes the model**. A different sklearn version, a tweak to feature engineering, an unintentional rerun of the GridSearch — any of those can produce a champion that differs from the one whose CV CI you reported at M3. Saving the joblib at M3 pins the model exactly; M4's test-set ceremony then evaluates the SAME model the M3 report claimed. The joblib is the contract between M3 and M4.

---

## 1. Setup — Imports, References, Helpers

Same imports you have seen across nb09–nb14, plus `joblib` (for saving the champion pipeline) and `json` (for `CONFIG.json`). The `cv_summary` helper returns mean, SD, half-width, and CI bounds in one call — every comparison in this notebook uses it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import joblib
from pathlib import Path
import datetime as dt
from scipy import stats

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold,
                                     cross_val_score, cross_val_predict,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             r2_score, mean_squared_error, mean_absolute_error,
                             confusion_matrix, ConfusionMatrixDisplay,
                             RocCurveDisplay, PrecisionRecallDisplay,
                             brier_score_loss)
from sklearn.calibration import calibration_curve

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'
GREEN     = '#2ca02c'
RED       = '#d62728'

# ----- M2 baselines (the Week-2 references) -----
reference_clf = Pipeline([('scaler', StandardScaler()),
                          ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))])
reference_reg = Pipeline([('scaler', StandardScaler()),
                          ('reg',    LinearRegression())])

# ----- Helper: CV with mean + 95% CI (Student's t, df=k-1) -----
def cv_summary(model, X, y, cv, scoring, k=5):
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    t_crit = stats.t.ppf(0.975, df=k - 1)
    mean = float(scores.mean()); sd = float(scores.std(ddof=1))
    half_w = t_crit * sd / np.sqrt(k)
    return {'mean': mean, 'sd': sd, 'half_w': half_w,
            'ci_low': mean - half_w, 'ci_high': mean + half_w, 'folds': scores}

print('✓ Setup complete — imports, seeds, M2 references, cv_summary helper.')

---

## 2. Load Both Datasets — Same Splits as nb11–nb14

Same 60/20/20 partitions, same `random_state=RANDOM_SEED`, same CV splitters. The CV scores you compute here chain directly to nb14's ceremony numbers.

In [ ]:
# CLASSIFICATION — Wisconsin Breast Cancer (MedScreen)
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target
X_clf_temp, X_test_clf, y_clf_temp, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# REGRESSION — California Housing (HomeValue Analytics)
data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target
X_reg_temp, X_test_reg, y_reg_temp, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification: train {len(X_train_clf):>5} | val {len(X_val_clf):>4} | test {len(X_test_clf):>4} (LOCKED until M4)')
print(f'Regression:     train {len(X_train_reg):>5} | val {len(X_val_reg):>4} | test {len(X_test_reg):>4} (LOCKED until M4)')

---

## 3. M3 §1a — Baseline Model Replication (M2)

The M3 rubric's first scored block is **§1a — Baseline Model (20 points)**. It asks you to **replicate your M2 baseline** on the training fold with k-fold CV and **report the headline metric with a 95% confidence interval**. Do not retune; do not feature-engineer further; just refit and report.

In our two demo cases the M2 baselines are the **Week-2 references** that survived nb09's tuning sweeps: `LogReg(C=1.0)` for classification and `OLS` for regression. Both wrapped in a `StandardScaler` pipeline.

In [ ]:
# Refit M2 baselines and compute 5-fold CV mean + 95% CI (Student's t, df=4)
clf_baseline = cv_summary(reference_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', k=5)
reg_baseline = cv_summary(reference_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      k=5)

print('=== M2 BASELINE — CLASSIFICATION (MedScreen, Wisconsin Breast Cancer) ===')
print(f"  Model: LogReg(C=1.0) inside StandardScaler pipeline")
print(f"  5-fold CV ROC-AUC: {clf_baseline['mean']:.4f}  (95% CI: [{clf_baseline['ci_low']:.4f}, {clf_baseline['ci_high']:.4f}])")
print(f"  fold scores: {np.round(clf_baseline['folds'], 4)}")
print()
print('=== M2 BASELINE — REGRESSION (HomeValue Analytics, California Housing) ===')
print(f"  Model: OLS inside StandardScaler pipeline")
print(f"  5-fold CV R²:      {reg_baseline['mean']:.4f}  (95% CI: [{reg_baseline['ci_low']:.4f}, {reg_baseline['ci_high']:.4f}])")
print(f"  fold scores: {np.round(reg_baseline['folds'], 4)}")

**Reading the output:**

Both baselines come in exactly where nb08-nb14 measured them. **MedScreen's M2 baseline lands at CV ROC-AUC ≈ 0.994 with a tight 95% CI** — a near-ceiling score on a linearly separable problem. **HomeValue's M2 baseline lands at CV R² ≈ 0.586 with a wider 95% CI** — OLS explains the bulk of the signal but plenty of room remains. These two numbers, with their CIs, are the **floor every M3 complex model has to clear by a CI-clear margin to earn displacement**.

> **A question that often comes up here:** *"why use exactly the same configuration as M2 instead of trying to improve the baseline?"* Because the M3 rubric scores the **gain from your complex model**, and the only honest measurement of that gain is *complex model CV CI* minus *original baseline CV CI*. If you re-tune the baseline at M3, the comparison stops being a baseline-vs-complex comparison and becomes two-tuned-models-vs-each-other. Keep the baseline frozen at its M2 configuration; let the complex model fight for displacement under the rule.

---

## 4. M3 §1b — Complex Model + Hyperparameter Tuning + Selection

The M3 rubric's largest block is **§1b — More Complex Model (35 points combined)**: model choice (8), hyperparameter tuning with CV (12), model selection + final training + saved artifact (10), comparison-vs-baseline narrative (5).

The walkthrough below picks the natural complex-model family for each demo case:

- **Classification:** `GradientBoostingClassifier` tuned over `(learning_rate, n_estimators, max_depth)`. Boosting is sequential, harder to tune than RF, and historically performs at-or-near top of the leaderboard on small tabular problems.
- **Regression:** `GradientBoostingRegressor` tuned over the same three dimensions. The 12,384-row training set gives the joint grid enough signal to separate good hyperparameters from bad ones cleanly.

For your project, pick whichever **complex family** is most appropriate for your dataset (RF, GBM, SVM, …) and justify the choice in your report under §1b model choice (8 points).

In [ ]:
# CLASSIFICATION GridSearch — small grid because Wisconsin's near-ceiling
clf_param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators':  [100, 200],
    'max_depth':     [2, 3],
}
clf_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_SEED),
    param_grid=clf_param_grid,
    cv=cv_clf, scoring='roc_auc', n_jobs=-1, return_train_score=False
)
clf_grid.fit(X_train_clf, y_train_clf)

print('=== CLASSIFICATION — GBM GridSearch (5-fold CV ROC-AUC) ===')
print(f"  Best params:    {clf_grid.best_params_}")
print(f"  Best CV ROC-AUC: {clf_grid.best_score_:.4f}")

# Compute 95% CI half-width for the best estimator
k = 5; t_crit = stats.t.ppf(0.975, df=k - 1)
clf_best_idx = clf_grid.best_index_
clf_best_folds = np.array([clf_grid.cv_results_[f'split{i}_test_score'][clf_best_idx] for i in range(k)])
clf_complex = {'mean': clf_best_folds.mean(), 'sd': clf_best_folds.std(ddof=1)}
clf_complex['half_w'] = t_crit * clf_complex['sd'] / np.sqrt(k)
clf_complex['ci_low']  = clf_complex['mean'] - clf_complex['half_w']
clf_complex['ci_high'] = clf_complex['mean'] + clf_complex['half_w']
clf_complex['folds']   = clf_best_folds
print(f"  CV ROC-AUC mean: {clf_complex['mean']:.4f}  (95% CI: [{clf_complex['ci_low']:.4f}, {clf_complex['ci_high']:.4f}])")

# REGRESSION GridSearch — larger grid because California Housing has more signal to chase
reg_param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
}
reg_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_SEED),
    param_grid=reg_param_grid,
    cv=cv_reg, scoring='r2', n_jobs=-1, return_train_score=False
)
reg_grid.fit(X_train_reg, y_train_reg)

print('\n=== REGRESSION — GBM GridSearch (5-fold CV R²) ===')
print(f"  Best params:    {reg_grid.best_params_}")
print(f"  Best CV R²:     {reg_grid.best_score_:.4f}")

reg_best_idx = reg_grid.best_index_
reg_best_folds = np.array([reg_grid.cv_results_[f'split{i}_test_score'][reg_best_idx] for i in range(k)])
reg_complex = {'mean': reg_best_folds.mean(), 'sd': reg_best_folds.std(ddof=1)}
reg_complex['half_w'] = t_crit * reg_complex['sd'] / np.sqrt(k)
reg_complex['ci_low']  = reg_complex['mean'] - reg_complex['half_w']
reg_complex['ci_high'] = reg_complex['mean'] + reg_complex['half_w']
reg_complex['folds']   = reg_best_folds
print(f"  CV R² mean:     {reg_complex['mean']:.4f}  (95% CI: [{reg_complex['ci_low']:.4f}, {reg_complex['ci_high']:.4f}])")

# Apply the CI-overlap rule for each spine
def ci_verdict(baseline, complex_):
    overlap = not (baseline['ci_high'] < complex_['ci_low'] or complex_['ci_high'] < baseline['ci_low'])
    if not overlap:
        winner = 'complex model' if complex_['mean'] > baseline['mean'] else 'M2 baseline'
        return f'CIs do NOT overlap → {winner} wins outright (CI-clear margin).'
    return 'CIs overlap → statistical tie → simpler M2 baseline wins by parsimony.'

print('\n=== CI-OVERLAP RULE — VERDICT PER CASE ===')
print(f'MedScreen (clf):   {ci_verdict(clf_baseline, clf_complex)}')
print(f'HomeValue (reg):   {ci_verdict(reg_baseline, reg_complex)}')

**Reading the output:**

The CI-overlap rule produces **two different verdicts** on the two demo cases — and that contrast itself is the M3 lesson.

**MedScreen (classification).** The tuned GBM's CV CI overlaps the M2 LogReg's CV CI heavily — both are pinned near the ceiling (ROC-AUC ≈ 0.99). Under the CI-overlap rule, the boosted ensemble has NOT earned displacement of the linear baseline, so **MedScreen's M3 champion stays `LogReg(C=1.0)`**. This is consistent with nb14's selection-ceremony verdict and is the honest finding to report — *"we tried a complex model; the data did not give it room to win."* That is a valid M3 outcome.

**HomeValue (regression).** The tuned GBM's CV CI sits well above the M2 OLS's CV CI with **no overlap**. The complex model is CI-clear better — earned displacement under the rule. **HomeValue's M3 champion is the tuned GBM** with the GridSearch's best hyperparameters. In USD terms this is roughly a 23 R² point lift over OLS, which translates into materially better property-value predictions.

> **A question that often comes up here:** *"what if my GridSearch's best CV score is only marginally above the baseline?"* Apply the CI-overlap rule mechanically — do not trust the means alone. A 0.01 R² lift could be inside the fold-to-fold noise. If the CIs overlap, your honest M3 report says *"the complex model did not earn displacement; M3 champion = M2 baseline."* Three things matter on the rubric for that case: (a) you ran a systematic search, (b) you documented the result honestly, (c) you applied the rule rather than chasing the higher mean.

> **A question that often comes up here:** *"can I use RandomizedSearchCV instead?"* Yes — and the M3 rubric explicitly permits it. Use `GridSearchCV` when your grid is small (under \~50 configurations); switch to `RandomizedSearchCV(..., n_iter=30)` when the grid would otherwise be too large to enumerate (e.g., when tuning four or more hyperparameters with broad ranges).

---

## 5. M3 §1b — Final Training + Save `champion_pipeline.joblib` + `CONFIG.json`

After the CI-overlap rule names the champion, the rubric requires three steps:

1. **Retrain** the champion on the **full training fold** (train + validation rows together — every drop of training signal goes into the deployed model).
2. **Save** the fitted pipeline as `champion_pipeline.joblib` so M4 can load it byte-for-byte.
3. **Save** a `CONFIG.json` recording the model family, hyperparameters, feature columns, and selection date. M4 uses it to verify the loaded pipeline matches what M3 reported.

The cells below do this on **both** demo cases. Your project needs to do it on **one** — whichever case your group is working.

In [ ]:
# ARTIFACTS_DIR is where you save the joblib + CONFIG.json — under your project folder in practice
ARTIFACTS_DIR = Path('m3_artifacts_demo')
ARTIFACTS_DIR.mkdir(exist_ok=True)

# ---- CLASSIFICATION ----
# Per CI-overlap verdict, MedScreen's champion is the M2 baseline (LogReg).
# We still retrain on the full training fold for symmetry with the regression case.
clf_champion = (Pipeline([('scaler', StandardScaler()),
                          ('clf', LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))])
                if clf_baseline['mean'] >= clf_complex['mean'] or
                   not (clf_baseline['ci_high'] < clf_complex['ci_low'] or
                        clf_complex['ci_high'] < clf_baseline['ci_low'])
                else clf_grid.best_estimator_)
# Combine train + val rows for final training (X_train_full_clf)
X_train_full_clf = pd.concat([X_train_clf, X_val_clf]); y_train_full_clf = pd.concat([y_train_clf, y_val_clf])
clf_champion.fit(X_train_full_clf, y_train_full_clf)
joblib.dump(clf_champion, ARTIFACTS_DIR / 'clf_champion_pipeline.joblib')

clf_config = {
    'case':                'MedScreen — Wisconsin Breast Cancer (classification)',
    'champion_family':     'LogReg(C=1.0)' if isinstance(clf_champion.named_steps.get('clf', None), LogisticRegression) else 'GradientBoostingClassifier (tuned)',
    'hyperparameters':     clf_grid.best_params_ if not isinstance(clf_champion.named_steps.get('clf', None), LogisticRegression) else {'C': 1.0, 'penalty': 'l2'},
    'feature_columns':     list(X_train_full_clf.columns),
    'cv_mean_roc_auc':     clf_baseline['mean'] if isinstance(clf_champion.named_steps.get('clf', None), LogisticRegression) else clf_complex['mean'],
    'cv_95ci_lo':          clf_baseline['ci_low']  if isinstance(clf_champion.named_steps.get('clf', None), LogisticRegression) else clf_complex['ci_low'],
    'cv_95ci_hi':          clf_baseline['ci_high'] if isinstance(clf_champion.named_steps.get('clf', None), LogisticRegression) else clf_complex['ci_high'],
    'random_seed':         RANDOM_SEED,
    'selection_date':      dt.date.today().isoformat(),
}
(ARTIFACTS_DIR / 'clf_CONFIG.json').write_text(json.dumps(clf_config, indent=2))
print(f"✓ Saved CLF artifacts: {ARTIFACTS_DIR}/clf_champion_pipeline.joblib + clf_CONFIG.json")
print(f"  Champion: {clf_config['champion_family']}")

# ---- REGRESSION ----
reg_champion = reg_grid.best_estimator_  # CI-clear win for the tuned GBM
X_train_full_reg = pd.concat([X_train_reg, X_val_reg]); y_train_full_reg = pd.concat([y_train_reg, y_val_reg])
reg_champion.fit(X_train_full_reg, y_train_full_reg)
joblib.dump(reg_champion, ARTIFACTS_DIR / 'reg_champion_pipeline.joblib')

reg_config = {
    'case':                'HomeValue Analytics — California Housing (regression)',
    'champion_family':     'GradientBoostingRegressor (tuned)',
    'hyperparameters':     reg_grid.best_params_,
    'feature_columns':     list(X_train_full_reg.columns),
    'cv_mean_r2':          reg_complex['mean'],
    'cv_95ci_lo':          reg_complex['ci_low'],
    'cv_95ci_hi':          reg_complex['ci_high'],
    'random_seed':         RANDOM_SEED,
    'selection_date':      dt.date.today().isoformat(),
}
(ARTIFACTS_DIR / 'reg_CONFIG.json').write_text(json.dumps(reg_config, indent=2))
print(f"\n✓ Saved REG artifacts: {ARTIFACTS_DIR}/reg_champion_pipeline.joblib + reg_CONFIG.json")
print(f"  Champion: {reg_config['champion_family']} (best params: {reg_config['hyperparameters']})")

**Reading the output:**

Two artifacts per case: a binary `*.joblib` (the fitted pipeline — sklearn can `joblib.load(...)` it byte-for-byte) and a human-readable `*_CONFIG.json` (model family, hyperparameters, feature columns, CV CI, selection date). Together they form the **M3→M4 contract**: M4's test-set ceremony will load the joblib and verify against `CONFIG.json` before opening the test envelope.

> **A question that often comes up here:** *"why retrain on the full training fold (train + validation) before saving?"* Because every drop of training signal counts at deployment time. The 60/20/20 split sets aside 20% of the data as a validation set for cross-validation discipline during model selection, but once the champion is named you no longer need a validation holdout — the CV CI is already computed. Refitting on train + val combined typically lifts test-set performance by a fraction of a point and is standard production practice. Your CV CI from §4 is what you report as the headline; the refit uses the same hyperparameters on more data.

---

## 6. M3 §1c — Required Visualizations

The M3 rubric §1c is worth **20 points** and lists three universally required figures plus a problem-type-specific set:

**Universal (every M3 submission needs all three):**

- **6.1 Hyperparameter-search plot** — CV metric vs. hyperparameter(s). Marks the selected best point.
- **6.2 Model-comparison bar chart** — two bars (M2 baseline vs. M3 champion), each with 95% CI error bars. This is the **visual evidence behind your CI-overlap-rule decision**.
- **6.3 Feature importance / coefficient plot** — top features for the champion model.

**Problem-type-specific:**

- **Regression:** predicted-vs-actual scatter + residual plot + **RMSE-by-quintile bar chart** (the decision-quality artifact for regression projects, in stakeholder units like USD).
- **Classification:** confusion matrix at the operating threshold + ROC + PR curves + optional **reliability diagram + Brier score** (the decision-quality artifact for classification projects).

The cells below generate every figure on both demo cases. Your M3 submission needs them on **your** case.

### 6.1 Hyperparameter-Search Plot

In [ ]:
# Pull `mean_test_score` for the regression GridSearch and arrange by (lr, n_estimators) for the depth that won.
best_depth = reg_grid.best_params_['max_depth']
cv_df = pd.DataFrame(reg_grid.cv_results_)
cv_df = cv_df[cv_df['param_max_depth'] == best_depth].copy()
heat = cv_df.pivot(index='param_learning_rate', columns='param_n_estimators', values='mean_test_score')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# CLF: line plot of CV ROC-AUC vs learning_rate at the depth that won
ax = axes[0]
clf_best_depth = clf_grid.best_params_['max_depth']
clf_df = pd.DataFrame(clf_grid.cv_results_)
clf_df = clf_df[clf_df['param_max_depth'] == clf_best_depth].copy()
for n_est in sorted(clf_df['param_n_estimators'].unique()):
    sub = clf_df[clf_df['param_n_estimators'] == n_est].sort_values('param_learning_rate')
    ax.plot(sub['param_learning_rate'], sub['mean_test_score'], marker='o',
            label=f'n_estimators={n_est}', linewidth=2)
# Mark the chosen best point
ax.scatter([clf_grid.best_params_['learning_rate']], [clf_grid.best_score_],
           color=GREEN, s=200, marker='*', zorder=5, label=f"best (CV={clf_grid.best_score_:.4f})")
ax.set_xlabel('learning_rate'); ax.set_ylabel('5-fold CV ROC-AUC')
ax.set_title(f'MedScreen — CV ROC-AUC vs learning_rate (max_depth={clf_best_depth})',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# REG: heatmap of CV R² over (lr, n_estimators) at the depth that won
ax = axes[1]
im = ax.imshow(heat.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns)
ax.set_yticks(range(len(heat.index)));   ax.set_yticklabels(heat.index)
ax.set_xlabel('n_estimators'); ax.set_ylabel('learning_rate')
ax.set_title(f'HomeValue — CV R² heatmap (max_depth={best_depth})', fontsize=12, fontweight='bold')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.values[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                color='white' if val > heat.values.mean() else 'black')
# Mark the chosen best cell
best_lr_idx = list(heat.index).index(reg_grid.best_params_['learning_rate'])
best_n_idx  = list(heat.columns).index(reg_grid.best_params_['n_estimators'])
ax.add_patch(plt.Rectangle((best_n_idx - 0.5, best_lr_idx - 0.5), 1, 1, fill=False, edgecolor='red', linewidth=3))
plt.colorbar(im, ax=ax, shrink=0.8, label='CV R²')

fig.suptitle('M3 §1c.6.1 — Hyperparameter-search plots (best points marked)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.2 Model-Comparison Bar Chart (M2 vs M3 with 95% CI Error Bars)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Classification
ax = axes[0]
labels_clf = ['M2 baseline\n(LogReg C=1.0)', 'M3 complex\n(tuned GBM)']
means_clf  = [clf_baseline['mean'],  clf_complex['mean']]
errs_clf   = [clf_baseline['half_w'], clf_complex['half_w']]
bars = ax.bar(labels_clf, means_clf, yerr=errs_clf, capsize=10,
              color=[GREY, CLF_COLOR], edgecolor='black')
for i, (m, e) in enumerate(zip(means_clf, errs_clf)):
    ax.text(i, m + e + 0.005, f'{m:.4f}', ha='center', fontsize=10)
ax.set_ylabel('5-fold CV ROC-AUC (mean ± 95% CI)')
ax.set_title('MedScreen — M2 vs M3', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Regression
ax = axes[1]
labels_reg = ['M2 baseline\n(OLS)', 'M3 complex\n(tuned GBM)']
means_reg  = [reg_baseline['mean'],  reg_complex['mean']]
errs_reg   = [reg_baseline['half_w'], reg_complex['half_w']]
ax.bar(labels_reg, means_reg, yerr=errs_reg, capsize=10,
       color=[GREY, REG_COLOR], edgecolor='black')
for i, (m, e) in enumerate(zip(means_reg, errs_reg)):
    ax.text(i, m + e + 0.005, f'{m:.4f}', ha='center', fontsize=10)
ax.set_ylabel('5-fold CV R² (mean ± 95% CI)')
ax.set_title('HomeValue — M2 vs M3', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('M3 §1c.6.2 — Model comparison (CV mean ± 95% CI)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.3 Feature Importance — Top Features for Each Champion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Classification: signed standardized coefficients from LogReg champion
ax = axes[0]
if isinstance(clf_champion.named_steps.get('clf', None), LogisticRegression):
    clf_coefs = clf_champion.named_steps['clf'].coef_.ravel()
    feat_names = X_train_full_clf.columns
    coef_df = pd.DataFrame({'feature': feat_names, 'coef': clf_coefs})
    coef_df = coef_df.reindex(coef_df['coef'].abs().sort_values(ascending=True).index).tail(10)
    colors = ['steelblue' if c > 0 else 'crimson' for c in coef_df['coef']]
    ax.barh(coef_df['feature'], coef_df['coef'], color=colors, edgecolor='black')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Standardized LogReg coefficient (signed)')
    ax.set_title('MedScreen — top 10 features by |coefficient|', fontsize=12, fontweight='bold')
else:
    perm = permutation_importance(clf_champion, X_train_full_clf, y_train_full_clf,
                                  scoring='roc_auc', n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1)
    pi_df = pd.DataFrame({'feature': X_train_full_clf.columns, 'imp': perm.importances_mean})
    pi_df = pi_df.sort_values('imp').tail(10)
    ax.barh(pi_df['feature'], pi_df['imp'], color=CLF_COLOR, edgecolor='black')
    ax.set_xlabel('Permutation importance (ROC-AUC drop)')
    ax.set_title('MedScreen — top 10 features by permutation importance', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Regression: permutation importance on the tuned GBM
ax = axes[1]
perm_reg = permutation_importance(reg_champion, X_train_full_reg, y_train_full_reg,
                                  scoring='r2', n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1)
pi_reg = pd.DataFrame({'feature': X_train_full_reg.columns, 'imp': perm_reg.importances_mean})
pi_reg = pi_reg.sort_values('imp').tail(10)
ax.barh(pi_reg['feature'], pi_reg['imp'], color=REG_COLOR, edgecolor='black')
ax.set_xlabel('Permutation importance (R² drop)')
ax.set_title('HomeValue — top 10 features by permutation importance', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

fig.suptitle('M3 §1c.6.3 — Feature importance for each champion', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.4 Regression Diagnostics — Predicted-vs-Actual + Residual + RMSE-by-Quintile

For regression projects, the M3 rubric requires these three figures (in addition to the universal three above). They are the **decision-quality artifacts** for regression: they show where the champion under- and over-predicts, and how prediction error scales across the target range.

In [ ]:
# Use out-of-fold predictions for honest diagnostics (no test-set touching)
y_pred_oof = cross_val_predict(reg_champion, X_train_full_reg, y_train_full_reg, cv=cv_reg, n_jobs=-1)
residuals = y_train_full_reg - y_pred_oof

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 6.4.a Predicted-vs-actual with y=x reference
ax = axes[0]
ax.scatter(y_train_full_reg, y_pred_oof, alpha=0.25, s=8, color=REG_COLOR)
lo, hi = float(min(y_train_full_reg.min(), y_pred_oof.min())), float(max(y_train_full_reg.max(), y_pred_oof.max()))
ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.6, label='Perfect prediction (y = x)')
ax.set_xlabel('Actual house value (100K USD)'); ax.set_ylabel('Predicted (CV out-of-fold)')
ax.set_title('Predicted vs Actual — HomeValue', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# 6.4.b Residual plot
ax = axes[1]
ax.scatter(y_pred_oof, residuals, alpha=0.25, s=8, color=REG_COLOR)
ax.axhline(0, color='black', linestyle='--', alpha=0.6)
ax.set_xlabel('Predicted house value (100K USD)'); ax.set_ylabel('Residual (actual − predicted)')
ax.set_title('Residual plot — HomeValue', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 6.4.c RMSE-by-quintile (decision-quality view in stakeholder units)
quintiles = pd.qcut(y_train_full_reg, q=5, labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)'])
rmse_by_q = []
for q in quintiles.unique():
    mask = quintiles == q
    rmse = np.sqrt(((y_train_full_reg[mask] - y_pred_oof[mask]) ** 2).mean())
    rmse_by_q.append({'quintile': q, 'rmse_USD': rmse * 100_000})
rmse_by_q = pd.DataFrame(rmse_by_q).set_index('quintile').reindex(['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)'])
ax = axes[2]
ax.bar(rmse_by_q.index, rmse_by_q['rmse_USD'], color=REG_COLOR, edgecolor='black')
for i, v in enumerate(rmse_by_q['rmse_USD']):
    ax.text(i, v + 1500, f'USD {v:,.0f}', ha='center', fontsize=10)
ax.set_ylabel('RMSE (USD)')
ax.set_title('RMSE-by-quintile — HomeValue (out-of-fold)', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=20)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('M3 §1c.6.4 — Regression diagnostics (HomeValue)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nRMSE-by-quintile table (out-of-fold predictions):')
print(rmse_by_q.round(0).to_string())

### 6.5 Classification Diagnostics — Confusion Matrix + ROC + PR + Reliability/Brier

For classification projects, the M3 rubric requires the confusion matrix + ROC + PR curves. The **reliability diagram + Brier score** are optional but recommended when the model's probabilities feed a downstream decision threshold (which most classification tools do).

In [ ]:
# Use out-of-fold predictions for honest diagnostics
y_pred_clf_oof  = cross_val_predict(clf_champion, X_train_full_clf, y_train_full_clf, cv=cv_clf, n_jobs=-1)
y_proba_clf_oof = cross_val_predict(clf_champion, X_train_full_clf, y_train_full_clf,
                                    cv=cv_clf, method='predict_proba', n_jobs=-1)[:, 1]

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 6.5.a Confusion matrix at default 0.5 threshold
ax = axes[0, 0]
cm = confusion_matrix(y_train_full_clf, y_pred_clf_oof)
ConfusionMatrixDisplay(cm, display_labels=['malignant (0)', 'benign (1)']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — MedScreen (OOF, threshold=0.5)', fontsize=12, fontweight='bold')

# 6.5.b ROC curve
ax = axes[0, 1]
RocCurveDisplay.from_predictions(y_train_full_clf, y_proba_clf_oof, ax=ax, color=CLF_COLOR)
ax.set_title('ROC curve — MedScreen (OOF)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 6.5.c PR curve
ax = axes[1, 0]
PrecisionRecallDisplay.from_predictions(y_train_full_clf, y_proba_clf_oof, ax=ax, color=CLF_COLOR)
ax.set_title('Precision-Recall curve — MedScreen (OOF)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 6.5.d Reliability diagram + Brier score
ax = axes[1, 1]
prob_true, prob_pred = calibration_curve(y_train_full_clf, y_proba_clf_oof, n_bins=10, strategy='quantile')
brier = brier_score_loss(y_train_full_clf, y_proba_clf_oof)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Perfect calibration')
ax.plot(prob_pred, prob_true, marker='o', color=CLF_COLOR, linewidth=2,
        label=f'Champion (Brier = {brier:.4f})')
ax.set_xlabel('Predicted probability (mean per bin)'); ax.set_ylabel('Empirical positive fraction')
ax.set_title('Reliability diagram — MedScreen (OOF)', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('M3 §1c.6.5 — Classification diagnostics (MedScreen)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f'Brier score (lower is better; 0 = perfect, 0.25 = uninformative on a 50/50 problem): {brier:.4f}')

**Reading the output:**

**MedScreen's diagnostics** confirm the M3 verdict from §4. The confusion matrix at the 0.5 threshold shows minimal errors; ROC sits near (0, 1) with AUC near 0.99; the PR curve hugs the upper-right corner; the reliability diagram tracks the diagonal closely with a Brier score under 0.05. **The M2 LogReg is a well-calibrated, high-quality classifier on this dataset**, and the M3 complex model offered no measurable lift — the honest M3 report says so.

**HomeValue's diagnostics** show the kind of structure a regression poster should highlight. The predicted-vs-actual scatter clusters around the y=x line but fans out at the top of the price range (the model under-predicts the most expensive properties — a classic ceiling effect when the training-set tail is sparse). The residual plot shows non-zero mean residuals at the high end, confirming the bias. The RMSE-by-quintile bar chart translates this into stakeholder language: **prediction error is roughly USD 40-50K in the middle quintiles but climbs to USD 80-90K in the top quintile.** That is the kind of finding HomeValue's deployment council will want to see — *"the model is reliable for typical homes but should be reviewed manually for properties above the 80th-percentile price."*

> **A question that often comes up here:** *"do I need to include all of 6.1 through 6.5 in my M3 report, or just the ones for my project type?"* The **three universal figures (6.1, 6.2, 6.3) are required** for every M3 submission. The problem-type-specific set is **either 6.4 (regression) OR 6.5 (classification)** — pick the one that matches your project. If your model's probabilities feed a downstream decision threshold (binary screening, risk tiering, etc.), the reliability diagram + Brier in 6.5 is worth including even though it is technically optional.

---

## 7. M3 §2 — Draft Abstract (\~250 Words)

The abstract is **worth 15 points** on the M3 rubric and becomes the **lead paragraph of the M4 poster**. Treat it like a press release for your project — every sentence pulls weight.

**The six required elements (in order):**

1. **Project title** — concise and informative. *If your dataset is synthetic, the title must say so explicitly.*
2. **Prediction problem** — framed as a question with a question mark. Example: *"Can six-month customer-churn risk be predicted from transaction history and engagement metrics?"*
3. **Prediction goal and motivation** — one or two sentences: what you predict and why it matters.
4. **Methodology and tools** — data prep, analytical methods (baseline + complex model + tuning + CV protocol), and tools (sklearn, etc.).
5. **Key findings / expected contributions** — preliminary results or anticipated contributions.
6. **Broader implications** — how the project informs business practice or contributes to the field of predictive analytics.

**Worked example — HomeValue Analytics regression case (draft):**

> ### Predicting California Property Values with Gradient Boosting: A 23 R²-Point Lift Over Linear Regression on the StatLib Census-Tract Dataset
>
> Can median property value at the census-tract level be predicted from demographic and geographic features available in the 1990 US Census, and does a tuned gradient-boosting machine outperform the conventional OLS baseline?
>
> Accurate property valuation underpins everything from tax assessment to lending decisions to underwriting. HomeValue Analytics' pricing team currently relies on linear regression with hand-engineered features, and the deployment council wants to know whether a more flexible model would materially improve the tool.
>
> We replicated the OLS baseline on 12,384 training tracts from the StatLib California Housing dataset and tuned a `GradientBoostingRegressor` across `(learning_rate, n_estimators, max_depth)` using 5-fold cross-validation inside `GridSearchCV`. Both models used identical preprocessing (`StandardScaler` inside an sklearn `Pipeline`) and the same 60/20/20 train/val/test split with `random_state=474`. Hyperparameter selection followed the CI-overlap rule from nb08.
>
> The tuned gradient-boosting champion reaches 5-fold CV R² ≈ 0.82 (95% CI [0.80, 0.84]) compared to the OLS baseline's CV R² ≈ 0.59 (95% CI [0.55, 0.62]). The two confidence intervals are disjoint — the complex model has earned displacement under the rule. RMSE-by-quintile diagnostics show prediction error of roughly USD 40-50K in middle quintiles, climbing to USD 80-90K in the top quintile (a ceiling effect worth flagging for human review on high-value properties).
>
> For HomeValue Analytics' pricing tool, gradient boosting is a defensible upgrade path with material downstream impact on assessment, lending, and underwriting decisions. The same workflow — baseline + complex model + tuning + CI-overlap rule + reliability diagnostics — generalizes to any tabular regression problem with comparable signal-to-noise structure.

*Word count: \~245.*

**Reading the abstract:** every required element is present, the prediction problem is framed as a question with a "?", the methodology paragraph names the search (`GridSearchCV` + 5-fold CV), the key-findings paragraph quotes both CIs and applies the CI-overlap rule, and the broader-implications paragraph anchors back to the stakeholder. The synthetic-data caveat is omitted because California Housing is real data; if your group's dataset is synthetic, the title must say so explicitly.

> **A question that often comes up here:** *"how many revisions should I plan for the abstract?"* The rubric guidance says **three rounds of revision is the floor, not the ceiling**. Draft modeling section first; abstract synthesizes after the numbers settle. Then have at least two teammates read it aloud and time it (a good 250-word abstract reads in about 90 seconds at a conference-friendly pace). If anything sounds clunky or unclear, rewrite.

---

## 8. Wrap-Up — M3 Submission Checklist + Bridge to nb16

**Before uploading `NN_complex_model.pdf` + `NN_complex_model.ipynb` to Brightspace, verify:**

- [ ] **§0 Prediction Goal** restated and regression-vs-classification confirmed
- [ ] **§1a M2 baseline** replicated with 5- or 10-fold CV and **95% CI** reported
- [ ] **§1b model choice** justified (why this complex family for this problem)
- [ ] **§1b hyperparameter tuning** documented (grid or distribution, k value, CV protocol)
- [ ] **§1b CI-overlap rule** applied honestly to pick the champion (or to keep the baseline)
- [ ] **§1b final training** on the full training fold and **`champion_pipeline.joblib`** + **`CONFIG.json`** saved
- [ ] **§1c.6.1 Hyperparameter-search plot** with best point marked
- [ ] **§1c.6.2 Model-comparison bar chart** with 95% CI error bars
- [ ] **§1c.6.3 Feature-importance plot** for the champion
- [ ] **§1c.6.4 OR 6.5** — regression diagnostics (predicted-vs-actual + residual + RMSE-by-quintile) OR classification diagnostics (confusion matrix + ROC + PR + optional reliability/Brier)
- [ ] **§2 Draft Abstract (\~250 words)** with all six required elements (title, prediction question, motivation, methodology, findings, broader implications)
- [ ] **Notebook runs top-to-bottom in Colab** with a fresh runtime (no manual cell skipping)

**Bridge to nb16 — Time-Series Forecasting.**

nb15 closed the loop on the cross-sectional modeling chain that started in nb01: from EDA → splits → metrics → linear → regularized linear → trees → forests → boosting → selection → champion → M3 deliverable. **nb16 introduces a different prediction setting: time series.** When the rows are ordered by time and "the future depends on the past," the k-fold CV machinery you learned in nb08 needs a different splitter (`TimeSeriesSplit`) and the feature-engineering vocabulary shifts to lag features, rolling windows, and walk-forward validation. The CV-CI rule survives the move — but the splits change.

> **A question that often comes up here:** *"do I need to read nb16 if my project is cross-sectional?"* Yes — nb16's lessons about temporal leakage transfer to any dataset where the rows have an order (transaction logs, sensor streams, monthly KPIs, even repeated surveys of the same customers). The mechanical CV-CI workflow you have now is the foundation; nb16 shows where that foundation needs reinforcement for time-aware problems.

---

<center>

**Thank you!**

</center>